[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adkipnis/metabeta/blob/main/demos/03-priors.ipynb)

# metabeta priors: specification, sensitivity and Bayes factors

metabeta's networks take the prior as an input, so a fit under another prior needs no
retraining, and several priors fit in one call. This notebook simulates a dataset with known
parameters, states priors in the schema `sample` accepts, compares the posteriors under three
priors fitted in one call, and computes Bayes factors with `evidence=True`. A CPU runtime is
enough; `01-quickstart` introduces the summary tables.

In [ ]:
%pip install -q metabeta==0.5.1

## Load a checkpoint

The response is continuous, so load the normal family's checkpoint.

In [1]:
import numpy as np
import pandas as pd
import torch

from metabeta import Api

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mb = Api.from_pretrained('normal', device=device)

Resolving metabeta 'normal' checkpoint from Hugging Face (adkipnis/metabeta@v1)...


Using checkpoint: /Users/alex/.cache/huggingface/hub/models--adkipnis--metabeta/snapshots/66c398ef31d65400f9ce6cff262265a9cd895a4d/metabeta-normal.pt


Loading metabeta checkpoint on cpu...


Metabeta checkpoint ready.


## Simulate the data

The data are simulated in plain numpy, so the true parameters are known. Students in 15
schools take a test; a student's `score` rises with their weekly study `hours`, at a rate that
varies from school to school, and does not depend on their `commute` in minutes. The model is
`score ~ hours + commute + (1 + hours | school)`: fixed effects of both predictors and, per
school, a random intercept and a random `hours` slope.

In [2]:
rng = np.random.default_rng(1)
n_schools = 15
school = np.repeat(np.arange(n_schools), rng.integers(6, 13, size=n_schools))  # (n,)
hours = rng.normal(5, 2, size=school.size).clip(0)
commute = rng.normal(30, 10, size=school.size).clip(5)

truth = pd.Series(
    {
        'Intercept': 60.0,
        'hours': 2.0,
        'commute': 0.0,
        'σ_Intercept': 6.0,
        'σ_hours': 1.0,
        'Corr(Intercept, hours)': -0.3,
        'σ_Residual': 8.0,
    }
)
sd_school = truth[['σ_Intercept', 'σ_hours']].to_numpy()
corr = np.array([[1, truth['Corr(Intercept, hours)']], [truth['Corr(Intercept, hours)'], 1]])
u = rng.multivariate_normal(np.zeros(2), corr * np.outer(sd_school, sd_school), size=n_schools)
score = (
    truth['Intercept']
    + u[school, 0]
    + (truth['hours'] + u[school, 1]) * hours
    + truth['commute'] * commute
    + rng.normal(0, truth['σ_Residual'], size=school.size)
)
df = pd.DataFrame(
    {'score': score, 'hours': hours, 'commute': commute, 'school': [f's{g:02d}' for g in school]}
)

sizes = df.groupby('school').size()
print(f'{len(df)} students in {len(sizes)} schools with {sizes.min()}–{sizes.max()} students each')
print(truth.to_string())

137 students in 15 schools with 6–12 students each
Intercept                 60.0
hours                      2.0
commute                    0.0
σ_Intercept                6.0
σ_hours                    1.0
Corr(Intercept, hours)    -0.3
σ_Residual                 8.0


## Specifying priors

`sample(priors=...)` takes a dictionary with up to four keys, one per kind of parameter.
Terms are named as in the formula, and a term or key left out keeps its default.

| key | parameters | per term | families | default |
|---|---|---|---|---|
| `fixed` | intercept and slopes | `{'nu': mean, 'tau': scale}` | `'normal'`, `'student'` | N(0, 2.5) |
| `random_sd` | SD of each random effect | `{'tau': scale}` | `'halfnormal'`, `'halfstudent'`, `'exponential'` | HN(2.5) |
| `sigma_eps` | residual SD | `{'tau': scale}`, without terms | as `random_sd` | HT₅(1) |
| `corr_rfx` | correlations of the random effects | `{'eta': concentration}`, without terms | LKJ | LKJ(1) |

A `'family'` entry sets the family of all the key's terms. The Student-t (t₅) and the
half-Student-t (HT₅) have five degrees of freedom, and an exponential prior's `tau` is its mean.
`sigma_eps` exists for the normal family only, and `corr_rfx` once there are two random effects.
An LKJ concentration above 1 favors weak correlations.

The priors are set on the standardized scale on which the networks were trained: `sample`
standardizes each numeric predictor and, for the normal family, the response to mean 0 and SD 1.
The defaults are [bambi](https://bambinos.github.io/bambi/)'s weakly informative priors on that
scale; N(0, 2.5) lets a one-SD change in a predictor move the response by several SDs. To state
a prior in the data's units, divide by the SDs `sample` standardizes with, as `slope` and
`points` do below.

Two priors besides the default:

- **informed** draws on a pilot study that reported about 1.5 points per hour of study: a
  Student-t centered there with scale 1 point per hour, and exponential priors with means of 5
  points for the SD of the school intercepts and 1 point per hour for the SD of the school
  slopes, a half-normal with scale 10 points for the residual SD and LKJ(2).
- **skeptical** doubts that study time matters: `hours` ~ N(0, 1 point per hour), all else at
  the defaults.

In [3]:
sd = df[['score', 'hours', 'commute']].std(ddof=0)  # the SDs sample standardizes with


def slope(term, value):
    """A slope of `value` points per unit of `term`, on the standardized scale."""
    return value * sd[term] / sd['score']


def points(value):
    """An SD of `value` points, on the standardized scale."""
    return value / sd['score']


informed = {
    'fixed': {'family': 'student', 'hours': {'nu': slope('hours', 1.5), 'tau': slope('hours', 1)}},
    'random_sd': {
        'family': 'exponential',
        'Intercept': {'tau': points(5)},
        'hours': {'tau': slope('hours', 1)},
    },
    'sigma_eps': {'family': 'halfnormal', 'tau': points(10)},
    'corr_rfx': {'eta': 2},
}
skeptical = {'fixed': {'hours': {'tau': slope('hours', 1)}}}
priors = {'default': None, 'informed': informed, 'skeptical': skeptical}

## Prior sensitivity in one call

A dictionary of named priors fits them all at once: `sample` stacks one row per prior and runs
the rows as one batch. `None` stands for the defaults.

In [4]:
%%time
torch.manual_seed(0)
formula = 'score ~ hours + commute + (1 + hours | school)'
result = mb.sample(df, formula=formula, priors=priors)

CPU times: user 217 ms, sys: 15.3 ms, total: 232 ms
Wall time: 161 ms


The result has one entry per row, in the order of the dictionary; all three rows went to the
same submodel:

In [5]:
result.routes

['normal-small', 'normal-small', 'normal-small']

`result.prior_params` holds the priors each row ran with, on the standardized scale: means ν,
scales τ and the LKJ concentration η. The default row shows ν = 0 and τ = 2.5 for every fixed
effect.

In [6]:
fixed, random = result.param_names['ffx'], result.param_names['sigma_rfx']
p = result.prior_params
standardized = pd.DataFrame(
    np.column_stack(
        [
            p['nu_ffx'][:, : len(fixed)],
            p['tau_ffx'][:, : len(fixed)],
            p['tau_rfx'][:, : len(random)],
            p['tau_eps'],
            p['eta_rfx'],
        ]
    ),
    index=list(priors),
    columns=[f'ν {t}' for t in fixed]
    + [f'τ {t}' for t in fixed]
    + [f'τ σ_{t}' for t in random]
    + ['τ σ_Residual', 'η Corr'],
)
standardized.T.round(3)

,default,informed,skeptical
ν Intercept,0.0,0.000,0.000
ν hours,0.0,0.283,0.000
ν commute,0.0,0.000,0.000
τ Intercept,2.5,2.500,2.500
τ hours,2.5,0.189,0.189
τ commute,2.5,2.500,2.500
τ σ_Intercept,2.5,0.530,2.500
τ σ_hours,2.5,0.189,2.500
τ σ_Residual,1.0,1.061,1.000
η Corr,1.0,2.000,1.000


### Posterior summaries

`posteriorSummary(result, batch_index=i)` prints row `i`. Its prior lines are in the data's
units. A normal prior on the intercept moves to `hours` = `commute` = 0 and collects the
slopes' priors there, so it widens. Lines ending in "at mean …" have no closed form at 0 and
state the prior where it was set, at the means of the listed predictors: the informed row's
Student-t intercept, and, because of the random `hours` slope, the SD of the school intercepts
and the correlation.

In [7]:
for i, name in enumerate(priors):
    print(f'── {name} ' + '─' * 60)
    print(mb.posteriorSummary(result, batch_index=i))
    print()

── default ────────────────────────────────────────────────────────────
Formula:  score ~ hours + commute + (1 + hours | school)
n = 137   m = 15
Priors:
  Intercept ~ N(71.2, 97.23)
  hours ~ N(0, 13.26)
  commute ~ N(0, 2.432)
  σ_Intercept ~ HN(23.57) at mean hours
  σ_hours ~ HN(13.26)
  Corr ~ LKJ(1) at mean hours
  σ_Residual ~ HT₅(9.426)

Fixed Effects:
|           |   Mean |    SD |   2.5% |   50.0% |   97.5% |   P(>0) |   Contr. |
|-----------|--------|-------|--------|---------|---------|---------|----------|
| Intercept | 62.917 | 2.968 | 57.283 |  63.044 |  68.888 |   1.000 |    0.999 |
| hours     |  2.105 | 0.392 |  1.344 |   2.098 |   2.900 |   1.000 |    0.999 |
| commute   | -0.064 | 0.067 | -0.196 |  -0.062 |   0.061 |   0.158 |    0.999 |

Standard Deviations:
|           |   Mean |    SD |   2.5% |   50.0% |   97.5% |
|-----------|--------|-------|--------|---------|---------|
| Intercept |  6.439 | 2.586 |  2.829 |   6.212 |  12.572 |
| hours     |  0.564 | 0.477 |

### Side by side

The table lines up the posterior means and 95% intervals of the three rows in the data's units,
next to the true values; `result.scale_info` maps the draws as the summaries do.

The fixed effects barely depend on the prior. The `hours` slope is 2.11 under the default prior,
2.01 under the informed one, whose center of 1.5 pulls it down a little, and 1.83 under the
skeptical one; every interval contains the true 2.0. In the skeptical row the contraction of
`hours` drops from 0.999 to 0.850: that prior is narrow enough to compete with the data.
`commute` stays at −0.06 with an interval around 0 under every prior. The SD of the school
slopes depends on the prior most, because 15 schools say little about how the slope varies:
the informed exponential prior pulls its mean from 0.56 to 0.38 and the upper end of its
interval from 1.60 to 1.21.

In [8]:
def estimates(result, i):
    """Posterior mean [95% interval] of row i's global parameters, in the data's units."""
    si, post = result.scale_info, result.proposal
    ffx = si.to_original_scale(post.ffx[i])[:, : len(fixed)]  # (S, d)
    sigma, _ = si.sdCorrToOriginal(post.sigma_rfx[i], post.corr_rfx[i])
    draws = torch.cat([ffx, sigma[:, : len(random)], post.sigma_eps[i, :, None]], -1).numpy()
    lo, hi = np.percentile(draws, [2.5, 97.5], axis=0)
    return [f'{m:.2f} [{l:.2f}, {h:.2f}]' for m, l, h in zip(draws.mean(0), lo, hi)]


names = fixed + [f'σ_{t}' for t in random] + ['σ_Residual']
side_by_side = pd.DataFrame(
    {name: estimates(result, i) for i, name in enumerate(priors)},
    index=pd.Index(names, name='mean [95% interval]'),
)
side_by_side.insert(0, 'truth', truth[names])
side_by_side

,truth,default,informed,skeptical
mean [95% interval],,,,
Intercept,60.0,"62.92 [57.28, 68.89]","63.21 [56.18, 69.05]","63.85 [57.95, 69.65]"
hours,2.0,"2.11 [1.34, 2.90]","2.01 [1.32, 2.68]","1.83 [1.07, 2.60]"
commute,0.0,"-0.06 [-0.20, 0.06]","-0.06 [-0.18, 0.07]","-0.06 [-0.19, 0.08]"
σ_Intercept,6.0,"6.44 [2.83, 12.57]","5.99 [3.23, 9.92]","6.43 [2.51, 11.61]"
σ_hours,1.0,"0.56 [0.02, 1.60]","0.38 [0.01, 1.21]","0.56 [0.03, 2.05]"
σ_Residual,8.0,"7.25 [6.40, 8.28]","7.36 [6.46, 8.55]","7.35 [6.49, 8.43]"


## Bayes factors

`evidence=True` adds `result.evidence`. Its `log_evidence` holds one value per row, the log
marginal likelihood log p(y): how probable the data are under the row's model, priors included.
The log Bayes factor of two models of the same data is the difference of their entries; there
is no separate field for it. `psis_k` is the Pareto k̂ diagnostic of the importance-sampling
estimate, and `sample` warns where it exceeds `psis_k_threshold`.

Does `commute` matter? Compare the model with it against the model without it. A Bayes factor
depends on the prior of any parameter only one of the models has, so fit the model with
`commute` under three priors for its slope, in one call: the default and two narrower normals,
stated in points per minute.

In [9]:
%%time
torch.manual_seed(0)
commute_priors = {
    'default': None,
    'medium': {'fixed': {'commute': {'tau': slope('commute', 0.5)}}},
    'narrow': {'fixed': {'commute': {'tau': slope('commute', 0.1)}}},
}
result_with = mb.sample(df, formula=formula, priors=commute_priors, evidence=True)

CPU times: user 208 ms, sys: 12 ms, total: 221 ms
Wall time: 146 ms


In [10]:
%%time
torch.manual_seed(0)
result_without = mb.sample(df, formula='score ~ hours + (1 + hours | school)', evidence=True)

CPU times: user 98.7 ms, sys: 9.19 ms, total: 108 ms
Wall time: 90.3 ms


In [11]:
result_without.evidence

{'log_evidence': array([-493.335124]),
 'psis_k': array([0.20391501]),
 'pool_psis_k': array([0.16921613]),
 'efficiency': array([0.53179395]),
 'psis_k_threshold': 0.7,
 'psis_flag': array([False])}

The log Bayes factor (BF) of the model without `commute` against the model with it is the
difference of their `log_evidence` entries, one per prior of the slope; above 0 it favors
leaving `commute` out. Its size reads on [Kass and Raftery](https://doi.org/10.1080/01621459.1995.10476572)'s scale.

In [12]:
def strength(log_bf):
    """How strongly a Bayes factor favors its model: Kass and Raftery (1995), JASA 90:773."""
    bf = np.exp(abs(log_bf))
    for bound, label in [(3, 'not worth more than a bare mention'), (20, 'positive'), (150, 'strong')]:
        if bf < bound:
            return label
    return 'very strong'


p_with = result_with.prior_params['tau_ffx'][:, fixed.index('commute')]
log_ev_with = result_with.evidence['log_evidence']
log_ev_without = result_without.evidence['log_evidence'][0]
log_bf = log_ev_without - log_ev_with
bayes_factors = pd.DataFrame(
    {
        'SD (points per minute)': p_with / slope('commute', 1),
        'log p(y) with commute': log_ev_with,
        'PSIS k̂': result_with.evidence['psis_k'],
        'log BF': log_bf,
        'BF': np.exp(log_bf),
        'evidence': [strength(b) for b in log_bf],
    },
    index=pd.Index(list(commute_priors), name='commute prior'),
)
print(f'log p(y) without commute: {log_ev_without:.2f}')
bayes_factors.round(2)

log p(y) without commute: -493.34


,SD (points per minute),log p(y) with commute,PSIS k̂,log BF,BF,evidence
commute prior,,,,,,
default,2.43,-496.49,0.22,3.16,23.54,strong
medium,0.50,-494.93,0.41,1.60,4.94,positive
narrow,0.10,-493.60,0.44,0.27,1.31,not worth more than a bare mention


Every Bayes factor favors leaving `commute` out, but by how much depends on the prior of its
slope. The default prior, with an SD of 2.43 points per minute, spreads the slope over values
the data rule out, and the evidence charges the model with `commute` for them: a Bayes factor
of 23.54, strong evidence. The narrower the prior, the closer the two models' predictions: at
0.1 points per minute the model with `commute` nearly is the model without it, and the Bayes
factor of 1.31 is not worth more than a bare mention. A Bayes factor thus compares the priors as much as the
predictors, so report it with the prior it came from. The rows of `result_with` differ only in
that prior, so their differences are Bayes factors between priors. Every PSIS k̂ is below 0.7,
so `sample` warned about none of them.

## Recap

- `priors=` takes a dictionary of `fixed`, `random_sd`, `sigma_eps` and `corr_rfx` on the
  standardized scale; divide by the data's SDs to state a prior in the data's units.
- A dictionary of named priors fits them all in one call, one row per prior;
  `posteriorSummary(result, batch_index=i)` prints row `i`.
- `evidence=True` returns `result.evidence['log_evidence']`, one log p(y) per row; a log
  Bayes factor is the difference of two entries for the same data, and it depends on the
  priors of the parameters the models do not share.